# 3-Class Forgery Classifier (Real / Edited / Deepfake)

Thin notebook -- logic lives in `data/*.py` and `model/*.py`. Sections: Data, Model, Train, Eval.
See `architecture_decisions.md` for the design and `data_download.md` / `model_code.md` for the implementation instructions this follows.

## Setup

In [ ]:
!pip install -q facenet-pytorch timm kaggle datasets

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
# Confirm GPU tier per model_code.md section 0 -- A100 gets the full 3-branch
# config below as-is; T4/L4 need the fallback config from architecture_decisions.md's GPU-tier table.

In [ ]:
# Mount Drive for persistent raw/processed data + checkpoints (recommended --
# without this, everything in data_raw/, dataset/, and checkpoints/ lives only on
# the ephemeral Colab VM disk and is lost when the runtime disconnects/recycles).
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DEEPFAKE_DATA_ROOT'] = '/content/drive/MyDrive/deepfake'  # must be set before any `config` import
# os.environ['KAGGLE_API_TOKEN'] = ''  # set before running the Data section
# os.environ['HF_TOKEN'] = ''          # optional, only needed if you hit HF rate limits

In [ ]:
# One-time: copy already-downloaded/face-filtered data from this session's local
# Colab disk (/content/deepfake/data_raw, dataset) to the Drive location config.py
# now points at -- so the download and face_filter cells above don't need to be
# re-run. Safe to re-run this cell: skips any folder that doesn't exist locally,
# and skips the copy entirely if the Drive destination already has content.
import shutil
from pathlib import Path

LOCAL_ROOT = Path('/content/deepfake')  # adjust if your repo isn't cloned here
DRIVE_ROOT = Path(os.environ['DEEPFAKE_DATA_ROOT'])

for name in ('data_raw', 'dataset', 'checkpoints'):
    src = LOCAL_ROOT / name
    dst = DRIVE_ROOT / name
    if not src.exists():
        print(f"skip {name}: no local {src}")
        continue
    if dst.exists() and any(dst.iterdir()):
        print(f"skip {name}: {dst} already has content")
        continue
    print(f"copying {src} -> {dst} ...")
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"done: {name}")

## Data
Downloads the three raw sources, then runs uniform MTCNN face detect+crop, split, and manifest generation.

In [ ]:
from data.download import test_sources

# Test cell: no-download validation of every source in use -- HF schema + person-caption
# hit rate for COCO_AI, Kaggle auth + file listing for CASIA. PS-Battles is skipped by
# default (skip_ps_battles=True): its Kaggle slug ships only a downloader script + TSVs,
# not actual images, so using it would mean running an unvetted script against ~2018-era
# URLs -- the exact link-rot risk it was supposed to avoid. edited stays CASIA-only.
# Raises SystemExit with a per-source breakdown if anything fails.
test_sources()

In [ ]:
from data.download import download_all

# Final cell: full download. COCO_AI and CASIA run concurrently (independent I/O-bound
# network pulls -- total wall-clock is roughly the slower of the two instead of the sum).
# PS-Battles stays skipped (see the test cell above for why). Only run this after the
# test cell above passes.
download_all(n_pairs=3000)

In [ ]:
import subprocess, sys

# Test cell: quick diagnostic pass over 300 images/class -- writes to a separate
# dataset_diag/ dir (doesn't touch the real dataset), skips the deepfake
# survival-floor check, and prints per-class decode/mtcnn/save timing plus
# GPU-batch-fragmentation stats. Confirms MTCNN/CUDA and the CASIA source are
# wired up correctly before paying for the full pass.
subprocess.run([sys.executable, 'data/face_filter.py', '--limit', '300'], check=True)

In [ ]:
import subprocess, sys

# Final cell: full run over every downloaded source. face_filter.py is a script
# (not just importable functions) -- run it directly so its per-source logging
# (seen/detected/rejected, timing, fragmentation) prints in full, including the
# deepfake survival-rate guard. Only run this after the test cell above passes.
subprocess.run([sys.executable, 'data/face_filter.py'], check=True)

In [ ]:
# Class balance check -- must feed the class-weighted loss below, never hardcoded proportions
import pandas as pd
from config import MANIFEST_PATH

manifest = pd.read_csv(MANIFEST_PATH)
print(manifest.groupby(['class', 'split']).size())

## Model
Instantiate the 3-branch fusion model and dry-run a single batch through the full pipeline before committing to the full training loop.

In [ ]:
from config import DEVICE, EMBED_DIM, CLASSES
from model.dataset import get_dataloader
from model.fusion import ForgeryClassifier

model = ForgeryClassifier(embed_dim=EMBED_DIM, num_classes=len(CLASSES)).to(DEVICE)

dry_loader = get_dataloader('train', batch_size=8, shuffle=True, num_workers=2)
batch = next(iter(dry_loader))
rgb = batch['rgb'].to(DEVICE)
fft_mag = batch['fft_mag'].to(DEVICE)
srm = batch['srm_residual'].to(DEVICE)

logits, gate_weights = model(rgb, fft_mag, srm)
print('logits:', logits.shape, 'gate_weights:', gate_weights.shape)
assert logits.shape == (8, len(CLASSES))
assert gate_weights.shape == (8, 3)
print('Dry run OK -- shapes check out.')

## Train

In [ ]:
from model.train import train

best_path, best_macro_f1 = train(
    epochs=18,
    batch_size=64,  # drop to 32 on OOM before touching resolution, per model_code.md section 4
    lr=3e-4,
    weight_decay=1e-4,
    num_workers=4,
    patience=5,
)
print(best_path, best_macro_f1)

## Eval
Macro-F1, per-class precision/recall/ROC-AUC, 3x3 confusion matrix, and the paired Grad-CAM + gate-weight explainability dump.

In [ ]:
from model.eval import load_model, compute_metrics, print_report, save_metrics, explainability_dump
from model.dataset import get_dataloader, ForgeryDataset

eval_model = load_model(str(best_path))
val_loader = get_dataloader('val', batch_size=64, shuffle=False, num_workers=4)
metrics = compute_metrics(eval_model, val_loader)
print_report(metrics)
save_metrics(metrics)  # writes eval/metrics_val_<timestamp>.json under DATA_ROOT (Drive if mounted)

In [ ]:
# Saves each Grad-CAM overlay as a PNG plus one explainability.json index under
# EVAL_DIR (Drive-backed if mounted) -- survives runtime recycle, and any sample
# can be re-visualized later via load_explainability_index() without re-running
# inference.
val_dataset = ForgeryDataset('val')
explainability_results = explainability_dump(eval_model, val_dataset, samples_per_class=3)

In [ ]:
# Visualize the saved Grad-CAM heatmaps alongside their gate weights (in-notebook,
# inline -- separate from the saved PNGs written by the explainability_dump cell above)
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, min(4, len(explainability_results)), figsize=(16, 4))
for ax, r in zip(axes, explainability_results[:4]):
    img = Image.open(r['path']).convert('RGB')
    ax.imshow(img)
    ax.imshow(r['gradcam'], cmap='jet', alpha=0.4)
    gate_str = ', '.join(f"{k}:{v:.2f}" for k, v in r['gate_weights'].items())
    ax.set_title(f"{r['true_class']}\n{gate_str}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# On-demand Grad-CAM for any single val-set index (or swap in your own image path
# + label below) -- doesn't require re-running the full explainability_dump cell.
from model.eval import grad_cam
from config import CLASSES

idx = 0  # change to inspect a different val sample
sample = val_dataset[idx]
true_class = val_dataset.df.iloc[idx]['class']

cam, gate = grad_cam(
    eval_model, sample['rgb'], sample['fft_mag'], sample['srm_residual'],
    target_class=CLASSES.index(true_class),
)

fig, ax = plt.subplots(figsize=(5, 5))
img = Image.open(sample['path']).convert('RGB')
ax.imshow(img)
ax.imshow(cam, cmap='jet', alpha=0.4)
gate_str = ', '.join(f"{k}:{v:.2f}" for k, v in gate.items())
ax.set_title(f"{sample['path']}\ntrue={true_class}  gate=[{gate_str}]", fontsize=9)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Browse a PAST explainability dump without re-running the model at all -- e.g. in a
# fresh Colab session, once Drive is mounted (cell-3) and DEEPFAKE_DATA_ROOT is set.
from model.eval import load_explainability_index

saved = load_explainability_index()  # reads eval/explainability.json
for r in saved:
    print(r['path'], '-', r['true_class'], '- gate:', r['gate_weights'], '- overlay:', r['overlay_path'])

# Show one saved overlay PNG directly (already-rendered image + heatmap + gate weights)
Image.open(saved[0]['overlay_path'])

In [ ]:
# Save-everything cell -- run any time to persist the current session's state to
# DATA_ROOT (Drive if mounted), without re-running data/train. Safe to re-run.
import torch
from pathlib import Path
from config import CHECKPOINT_DIR, DATA_ROOT, DEVICE, EMBED_DIM, CLASSES
from model.eval import compute_metrics, print_report, save_metrics, explainability_dump
from model.dataset import get_dataloader, ForgeryDataset
from model.fusion import ForgeryClassifier

# 1. Data: nothing to do -- data_raw/ and dataset/ are written directly under
# DATA_ROOT by download.py/face_filter.py, so they're already wherever DATA_ROOT
# points (Drive, if mounted). Confirm:
print(f"DATA_ROOT: {DATA_ROOT}  (exists: {DATA_ROOT.exists()})")

# 2. Model: save the current in-memory model if it isn't already checkpointed
# (e.g. train() was interrupted, or you trained manually outside train()).
current_model = globals().get('eval_model') or globals().get('model')
if current_model is None:
    print("skip model save: no `model` or `eval_model` in scope")
else:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    current_path = CHECKPOINT_DIR / "current_model.pt"
    torch.save({"model_state_dict": current_model.state_dict()}, current_path)
    print(f"saved current model -> {current_path}")

# 3. Metrics: recompute + save if we have a model and val loader; otherwise
# save whatever was last computed in this session (`metrics`, from cell-16).
if current_model is not None:
    val_loader = get_dataloader('val', batch_size=64, shuffle=False, num_workers=4)
    current_metrics = compute_metrics(current_model, val_loader)
    print_report(current_metrics)
    save_metrics(current_metrics, tag="current")
elif 'metrics' in globals():
    save_metrics(metrics, tag="current")
else:
    print("skip metrics save: no model or existing `metrics` in scope")

# 4. Explainability dump: (re)run + save against the current model.
if current_model is not None:
    current_val_dataset = globals().get('val_dataset') or ForgeryDataset('val')
    explainability_dump(current_model, current_val_dataset, samples_per_class=3)
else:
    print("skip explainability dump: no model in scope")